# AI-Powered Intrusion Detection System (IDS)

## Objective
The goal of this project is to develop an AI-based IDS capable of detecting network intrusions, specifically DDoS attacks, using machine learning models.

This project evaluates:
- Random Forest (supervised learning)
- Isolation Forest (anomaly detection)
- A hybrid combination of both models

In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

## Data Loading

We use the CICIDS2017 dataset. Multiple files are combined for training, while a portion of DDoS traffic is reserved for testing.

In [3]:
train_files = [
    r"C:\Users\silva\Documents\ai_ids\datasets\MachineLearningCVE\Monday-WorkingHours.pcap_ISCX.csv",
    r"C:\Users\silva\Documents\ai_ids\datasets\MachineLearningCVE\Tuesday-WorkingHours.pcap_ISCX.csv",
    r"C:\Users\silva\Documents\ai_ids\datasets\MachineLearningCVE\Wednesday-workingHours.pcap_ISCX.csv",
    r"C:\Users\silva\Documents\ai_ids\datasets\MachineLearningCVE\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    r"C:\Users\silva\Documents\ai_ids\datasets\MachineLearningCVE\Friday-WorkingHours-Morning.pcap_ISCX.csv",
    r"C:\Users\silva\Documents\ai_ids\datasets\MachineLearningCVE\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv"
]

train_dfs = [pd.read_csv(f) for f in train_files]
df_base_train = pd.concat(train_dfs, ignore_index=True)

df_ddos = pd.read_csv(
    r"C:\Users\silva\Documents\ai_ids\datasets\MachineLearningCVE\ddos.csv"
)

print("Base train shape:", df_base_train.shape)
print("Full DDoS shape:", df_ddos.shape)

Base train shape: (2316396, 79)
Full DDoS shape: (225745, 79)


## Data Preprocessing

- Removed NaN and infinite values
- Standardized column names
- Created binary labels:
  - 0 = BENIGN
  - 1 = ATTACK

In [4]:
for df in [df_base_train, df_ddos]:
    df.columns = df.columns.str.strip()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    df["binary_label"] = df["Label"].apply(lambda x: 0 if x == "BENIGN" else 1)

## Dataset Strategy

To ensure realistic evaluation:

- Training data includes multiple attack types and a portion of DDoS traffic
- Testing data contains unseen DDoS traffic

This allows evaluation of both:
- Generalization capability
- Performance on known attack patterns

In [5]:
df_ddos_train, df_ddos_test = train_test_split(
    df_ddos,
    test_size=0.3,
    random_state=42,
    stratify=df_ddos["binary_label"]
)

print("DDoS train shape:", df_ddos_train.shape)
print("DDoS test shape:", df_ddos_test.shape)

DDoS train shape: (157997, 80)
DDoS test shape: (67714, 80)


In [6]:
df_train = pd.concat([df_base_train, df_ddos_train], ignore_index=True)
df_test = df_ddos_test.copy()

print("Final train shape:", df_train.shape)
print("Final test shape:", df_test.shape)

Final train shape: (2471767, 80)
Final test shape: (67714, 80)


In [7]:
print("Train labels:")
print(df_train["binary_label"].value_counts())

print("\nTest labels:")
print(df_test["binary_label"].value_counts())

Train labels:
binary_label
0    1953655
1     518112
Name: count, dtype: int64

Test labels:
binary_label
1    38408
0    29306
Name: count, dtype: int64


## Feature Selection

The following network flow features were selected for modeling:

- Destination Port
- Flow Duration
- Total Forward/Backward Packets
- Packet Length Mean
- Flow Bytes/s
- Flow Packets/s

These features capture traffic behavior relevant to DDoS detection.

In [8]:
features = [
    "Destination Port",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Mean",
    "Flow Bytes/s",
    "Flow Packets/s"
]

In [9]:
X_train = df_train[features].copy()
y_train = df_train["binary_label"].copy()

X_test = df_test[features].copy()
y_test = df_test["binary_label"].copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (2471767, 8)
X_test shape: (67714, 8)


## Baseline Random Forest Model

This model uses default threshold (0.5) and serves as a comparison point.

In [10]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [11]:
rf_preds = rf_model.predict(X_test)

In [12]:
print("Random Forest Accuracy:", accuracy_score(y_test, rf_preds))
print(confusion_matrix(y_test, rf_preds))
print(classification_report(y_test, rf_preds, zero_division=0))

Random Forest Accuracy: 0.9967362731488318
[[29118   188]
 [   33 38375]]
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     29306
           1       1.00      1.00      1.00     38408

    accuracy                           1.00     67714
   macro avg       1.00      1.00      1.00     67714
weighted avg       1.00      1.00      1.00     67714



## Tuned Random Forest Model (Final)

This model includes:
- Class balancing
- Increased number of trees
- Threshold tuning to improve attack detection

In [13]:
rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_preds_tuned = (rf_probs > 0.4).astype(int)

print("Tuned RF Accuracy:", accuracy_score(y_test, rf_preds_tuned))
print(confusion_matrix(y_test, rf_preds_tuned))
print(classification_report(y_test, rf_preds_tuned, zero_division=0))

Tuned RF Accuracy: 0.9965738252060136
[[29103   203]
 [   29 38379]]
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     29306
           1       0.99      1.00      1.00     38408

    accuracy                           1.00     67714
   macro avg       1.00      1.00      1.00     67714
weighted avg       1.00      1.00      1.00     67714



In [14]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Isolation Forest (Anomaly Detection)

This model is trained only on normal traffic to detect anomalous behavior.

In [15]:
X_train_normal = X_train_scaled[y_train == 0]

iso_model = IsolationForest(
    n_estimators=100,
    contamination=0.1,
    random_state=42
)

iso_model.fit(X_train_normal)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.1
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [16]:
iso_preds = iso_model.predict(X_test_scaled)
iso_preds_binary = [0 if pred == 1 else 1 for pred in iso_preds]

In [17]:
print("Isolation Forest Accuracy:", accuracy_score(y_test, iso_preds_binary))
print(confusion_matrix(y_test, iso_preds_binary))
print(classification_report(y_test, iso_preds_binary, zero_division=0))

Isolation Forest Accuracy: 0.46557580411731697
[[22291  7015]
 [29173  9235]]
              precision    recall  f1-score   support

           0       0.43      0.76      0.55     29306
           1       0.57      0.24      0.34     38408

    accuracy                           0.47     67714
   macro avg       0.50      0.50      0.44     67714
weighted avg       0.51      0.47      0.43     67714



## Hybrid Model

A combined model was created where traffic is classified as an attack if either:
- Random Forest predicts an attack, OR
- Isolation Forest flags it as anomalous

This approach increases sensitivity but may increase false positives.

In [18]:
final_preds = [
    1 if (rf == 1 or iso == 1) else 0
    for rf, iso in zip(rf_preds_tuned, iso_preds_binary)
]

In [19]:
print("Combined Model Accuracy:", accuracy_score(y_test, final_preds))
print(confusion_matrix(y_test, final_preds))
print(classification_report(y_test, final_preds, zero_division=0))

Combined Model Accuracy: 0.8930058776619311
[[22089  7217]
 [   28 38380]]
              precision    recall  f1-score   support

           0       1.00      0.75      0.86     29306
           1       0.84      1.00      0.91     38408

    accuracy                           0.89     67714
   macro avg       0.92      0.88      0.89     67714
weighted avg       0.91      0.89      0.89     67714



## Saving Results

The outputs of each model are saved for further analysis and easier reproduction

In [20]:
import os 
output_dir = "../outputs/ddos_model"
os.makedirs(output_dir, exist_ok=True)

In [21]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
rf_report = classification_report(y_test, rf_preds_tuned, zero_division=0)
rf_cm = confusion_matrix(y_test, rf_preds_tuned)
rf_acc = accuracy_score(y_test, rf_preds_tuned)

with open(f"{output_dir}/rf_results.txt", "w") as f:
    f.write(f"Random Forest Accuracy: {rf_acc}\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(rf_cm) + "\n\n")
    f.write("Classification Report:\n")
    f.write(rf_report)

In [22]:
iso_report = classification_report(y_test, iso_preds_binary, zero_division=0)
iso_cm = confusion_matrix(y_test, iso_preds_binary)
iso_acc = accuracy_score(y_test, iso_preds_binary)

with open(f"{output_dir}/iso_results.txt", "w") as f:
    f.write(f"Isolation Forest Accuracy: {iso_acc}\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(iso_cm) + "\n\n")
    f.write("Classification Report:\n")
    f.write(iso_report)

In [23]:
combined_report = classification_report(y_test, final_preds, zero_division=0)
combined_cm = confusion_matrix(y_test, final_preds)
combined_acc = accuracy_score(y_test, final_preds)

with open(f"{output_dir}/combined_results.txt", "w") as f:
    f.write(f"Combined Model Accuracy: {combined_acc}\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(combined_cm) + "\n\n")
    f.write("Classification Report:\n")
    f.write(combined_report)
    
    

In [24]:
import os

model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)

In [25]:
joblib.dump(rf_model, f"{model_dir}/ddos_rf_model.pkl")

['../models/ddos_rf_model.pkl']

In [27]:
joblib.dump(scaler, f"{model_dir}/ddos_scaler.pkl")

['../models/ddos_scaler.pkl']

## Model Comparison

| Model              | Accuracy | Precision (Attack) | Recall (Attack) |
|-------------------|---------|-------------------|----------------|
| Random Forest     | 0.9967  | 1.00              | 1.00           |
| Tuned RF (Final)  | 0.9966  | 0.99              | 1.00           |
| Isolation Forest  | 0.4656  | 0.57              | 0.24           |
| Hybrid Model      | 0.8930  | 0.84              | 1.00           |

## Conclusion

- Random Forest achieved the highest performance when trained with attack-specific data
- Isolation Forest struggled to detect structured DDoS attacks
- The hybrid model improved detection sensitivity but increased false positives
- Dataset composition significantly impacts IDS performance

The tuned Random Forest model was selected as the final IDS model due to its strong balance between precision and recall.